<a href="https://colab.research.google.com/github/amita-kapoor/Agentic-Systems-Engineering/blob/main/Chapter06/Compliance_agent_ch_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Listing 6.13 Environment setup


In [ ]:
# Install dependencies (uncomment in Colab on first run)
#!pip install openai pydantic --quiet

In [ ]:
import os
import json
import time
import hashlib
from typing import Any, Callable, Literal, Optional, TypedDict
from dataclasses import dataclass, field
from enum import Enum
from pydantic import BaseModel, Field
from google.colab import userdata

# Set your API key here, or leave as empty string to use the mock client.
try:
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

USE_REAL_LLM = bool(OPENAI_API_KEY)
print(f"Using real LLM: {USE_REAL_LLM}")

Using real LLM: True


### Listing 6.14 A unified LLM call interface


In [ ]:
def _mock_llm(system: str, user: str, want_json: bool) -> str:
    """Deterministic stand-in for an LLM. Returns plausible shapes, not smart answers."""
    seed = hashlib.sha256((system + user).encode()).hexdigest()
    if not want_json:
        return f"[mock-llm] response keyed on {seed[:8]}"
    # Detect what kind of JSON the caller wants from cues in the prompt.
    if "draft a compliance report" in user.lower() or "draft a report" in user.lower():
        return json.dumps({
            "summary": "The product processes customer payment data.",
            "clauses": [
                {"id": "REG-1", "claim": "Encrypts data at rest.", "evidence": "uses AES-256"},
                {"id": "REG-2", "claim": "Logs access events.", "evidence": "audit log"},
            ],
            "open_issues": [],
        })
    if "critique" in user.lower() or "evaluate" in user.lower():
        # Alternate between needs_revision and acceptable so the loop terminates.
        status = "needs_revision" if seed[0] in "0123456789ab" else "acceptable"
        return json.dumps({
            "status": status,
            "issues": ["missing cross-reference for REG-3"] if status == "needs_revision" else [],
            "feedback": "Add an explicit reference to clause REG-3 on data retention.",
        })
    return json.dumps({"result": f"mock-{seed[:8]}"})


def llm_call(system: str, user: str, schema: Optional[dict] = None,
             temperature: float = 0.2) -> Any:
    """Single entry point for all model calls in this notebook."""
    want_json = schema is not None
    if not USE_REAL_LLM:
        text = _mock_llm(system, user, want_json)
    else:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        kwargs = {
            "model": "gpt-4o-mini",
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            "temperature": temperature,
        }
        if want_json:
            kwargs["response_format"] = {"type": "json_object"}
        resp = client.chat.completions.create(**kwargs)
        text = resp.choices[0].message.content
    if want_json:
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return {"_raw": text, "_parse_error": True}
    return text


# Smoke test
print(llm_call("you are a test", "say hi", schema=None))

Hi! How can I assist you today?


### Listing 6.15 Semantic memory: the regulatory framework


In [ ]:
class RegulatoryClause(BaseModel):
    """A single regulatory requirement. Lives in semantic memory."""
    id: str
    title: str
    text: str
    required_evidence: list[str] = Field(default_factory=list)


class RegulatoryFramework:
    """Semantic memory for regulatory rules. Stable across tasks."""

    def __init__(self, clauses: list[RegulatoryClause]):
        self._clauses = {c.id: c for c in clauses}

    def get(self, clause_id: str) -> Optional[RegulatoryClause]:
        return self._clauses.get(clause_id)

    def all_ids(self) -> list[str]:
        return list(self._clauses.keys())

    def retrieve_relevant(self, product_description: str) -> list[RegulatoryClause]:
        """Stand-in for staged retrieval. Returns all clauses for this small example."""
        return list(self._clauses.values())

    def as_prompt_block(self, clauses: list[RegulatoryClause]) -> str:
        lines = []
        for c in clauses:
            evidence = ", ".join(c.required_evidence) or "none"
            lines.append(f"[{c.id}] {c.title}: {c.text} (required evidence: {evidence})")
        return "\n".join(lines)


# A small synthetic framework for a hypothetical "Customer Payment Data Directive"
FRAMEWORK = RegulatoryFramework([
    RegulatoryClause(
        id="REG-1",
        title="Encryption at rest",
        text="Customer payment data must be encrypted at rest using an industry-standard cipher.",
        required_evidence=["cipher_name", "key_management_policy"],
    ),
    RegulatoryClause(
        id="REG-2",
        title="Access logging",
        text="All access to payment data must be logged with user identity and timestamp.",
        required_evidence=["log_destination", "retention_period"],
    ),
    RegulatoryClause(
        id="REG-3",
        title="Data retention",
        text="Payment data must not be retained beyond 90 days unless explicitly justified.",
        required_evidence=["retention_period", "justification_if_exceeded"],
    ),
    RegulatoryClause(
        id="REG-4",
        title="Cross-border transfer",
        text="Cross-border transfers of payment data require a documented legal basis.",
        required_evidence=["transfer_destinations", "legal_basis"],
    ),
])

print("Framework loaded. Clause IDs:", FRAMEWORK.all_ids())

Framework loaded. Clause IDs: ['REG-1', 'REG-2', 'REG-3', 'REG-4']




### Listing 6.16 Episodic memory: remembering what worked

In [ ]:
@dataclass
class EpisodicRecord:
    task_signature: str          # short label for the kind of task
    outcome: Literal["success", "failure"]
    insight: str                 # what the agent learned
    timestamp: float = field(default_factory=time.time)


class EpisodicMemory:
    """Distilled records from past tasks. Grows over time, retrieved selectively."""

    def __init__(self) -> None:
        self._records: list[EpisodicRecord] = []

    def store(self, record: EpisodicRecord) -> None:
        self._records.append(record)

    def retrieve(self, task_signature: str, k: int = 3) -> list[EpisodicRecord]:
        """Score records by token overlap with the task signature."""
        if not self._records:
            return []
        query_tokens = set(task_signature.lower().split())
        scored = []
        for rec in self._records:
            rec_tokens = set(rec.task_signature.lower().split())
            overlap = len(query_tokens & rec_tokens)
            scored.append((overlap, rec))
        scored.sort(key=lambda x: (-x[0], -x[1].timestamp))
        return [rec for score, rec in scored[:k] if score > 0]

    def as_prompt_block(self, records: list[EpisodicRecord]) -> str:
        if not records:
            return "No relevant past experience."
        return "\n".join(f"- [{r.outcome}] {r.task_signature}: {r.insight}" for r in records)


MEMORY = EpisodicMemory()
print("Episodic memory initialized. Currently empty.")

Episodic memory initialized. Currently empty.




### Listing 6.17 The observation layer: a deterministic rule validator

In [ ]:
class ReportClause(BaseModel):
    """One assertion in the draft report, tied to a regulatory clause."""
    id: str
    claim: str
    evidence: str
    compliance_status: Literal["compliant", "non_compliant", "insufficient_information"]


class DraftReport(BaseModel):
    summary: str
    clauses: list[ReportClause]
    open_issues: list[str] = Field(default_factory=list)


class ValidationSignal(BaseModel):
    missing_clauses: list[str] = Field(default_factory=list)
    clauses_without_evidence: list[str] = Field(default_factory=list)
    unknown_clause_refs: list[str] = Field(default_factory=list)
    coverage_ratio: float = 0.0
    is_clean: bool = False


class RuleValidator:
    """Structural check. Now also verifies that 'compliant' claims cite required evidence."""

    MIN_EVIDENCE_LEN = 10

    def __init__(self, framework: RegulatoryFramework):
        self.framework = framework

    def validate(self, draft: DraftReport) -> ValidationSignal:
        all_required = set(self.framework.all_ids())
        addressed = {rc.id for rc in draft.clauses if rc.id in all_required}
        unknown = [rc.id for rc in draft.clauses if rc.id not in all_required]
        missing = sorted(all_required - addressed)
        no_evidence = [
            rc.id for rc in draft.clauses
            if rc.id in all_required and len(rc.evidence.strip()) < self.MIN_EVIDENCE_LEN
        ]

        # New: check that "compliant" claims cite at least one required evidence keyword.
        unsupported_compliant = []
        for rc in draft.clauses:
            if rc.id not in all_required:
                continue
            if rc.compliance_status != "compliant":
                continue
            clause = self.framework.get(rc.id)
            if not clause or not clause.required_evidence:
                continue
            evidence_lower = rc.evidence.lower()
            cited = [
                key for key in clause.required_evidence
                if any(token in evidence_lower for token in key.lower().split("_"))
            ]
            if not cited:
                unsupported_compliant.append(rc.id)

        coverage = len(addressed) / max(len(all_required), 1)
        is_clean = (
            not missing and not no_evidence and not unknown and not unsupported_compliant
        )

        signal = ValidationSignal(
            missing_clauses=missing,
            clauses_without_evidence=no_evidence,
            unknown_clause_refs=unknown,
            coverage_ratio=round(coverage, 2),
            is_clean=is_clean,
        )
        # We piggyback the new check on clauses_without_evidence for compatibility,
        # but in a real system this would be its own field with its own remediation flow.
        signal.clauses_without_evidence = list(set(no_evidence + unsupported_compliant))
        return signal

VALIDATOR = RuleValidator(FRAMEWORK)



### Listing 6.18 The generator: drafting a report

In [ ]:
GENERATOR_SYSTEM = """You are a compliance analyst.
Given a product description, regulatory clauses, and any prior feedback,
produce a structured compliance report.

Return JSON matching exactly:
{
  "summary": string,
  "clauses": [
    {
      "id": string,
      "claim": string,
      "evidence": string,
      "compliance_status": "compliant" | "non_compliant" | "insufficient_information"
    }, ...
  ],
  "open_issues": [string, ...]
}

Rules:
- "id" must match one of the provided regulatory clause IDs. Address every clause.
- "claim" restates what the regulation requires.
- "evidence" cites specific facts from the product description. At least 10 characters.
- "compliance_status":
    "compliant" only when the product description provides concrete facts that satisfy the clause.
    "non_compliant" when the description shows the requirement is violated.
    "insufficient_information" when the description does not say enough to judge.
- Do NOT mark a clause "compliant" by inventing facts. If unsure, use "insufficient_information".
- "open_issues" must list every clause that is non_compliant or insufficient_information,
  written as a plain English sentence that names the clause ID. Example:
  "Data retention practices are not documented (REG-3)."
  Do NOT use machine tags or @-syntax in open_issues.
"""


def generator(task: dict) -> DraftReport:
    rel_clauses = task["clauses"]
    parts = [
        f"PRODUCT DESCRIPTION:\n{task['product_description']}",
        f"\nREGULATORY CLAUSES:\n{FRAMEWORK.as_prompt_block(rel_clauses)}",
        f"\nPAST EXPERIENCE:\n{task['memory_block']}",
    ]
    if task.get("previous_draft"):
        prev = task["previous_draft"].model_dump()
        parts.append(f"\nPREVIOUS DRAFT:\n{json.dumps(prev, indent=2)}")
    if task.get("validator_signal"):
        parts.append(f"\nVALIDATOR SIGNAL:\n{json.dumps(task['validator_signal'].model_dump(), indent=2)}")
    if task.get("critique_feedback"):
        parts.append(f"\nCRITIQUE FEEDBACK:\n{task['critique_feedback']}")
    parts.append("\nDraft the report now.")

    raw = llm_call(GENERATOR_SYSTEM, "\n".join(parts), schema={"type": "object"})
    try:
        clauses = [ReportClause(**c) for c in raw.get("clauses", [])]
        return DraftReport(
            summary=raw.get("summary", ""),
            clauses=clauses,
            open_issues=raw.get("open_issues", []),
        )
    except Exception as e:
        return DraftReport(summary=f"[parse_error: {e}]", clauses=[], open_issues=[])



### Listing 6.19 The critic: combining rule signals with model judgment

In [ ]:
class Critique(TypedDict):
    status: Literal["acceptable", "needs_revision", "converged"]
    issues: list[str]
    feedback: str


# A whitelist of fields a blocker is allowed to cite. Anything else is treated as a suggestion.
CITABLE_FIELDS = {"summary", "clauses", "open_issues"}

# A whitelist of rule names. Maps to the same four blocker categories from before.
BLOCKER_RULES = {
    "evidence_status_mismatch",   # status contradicts evidence
    "summary_misrepresents",      # summary contradicts per-clause statuses
    "open_issues_incomplete",     # open_issues missing a non_compliant or insufficient clause
    "factual_error",              # contradicts product description or clause text
}


CRITIC_SYSTEM = f"""You are a senior compliance reviewer.

You output structured blockers. A blocker has three required fields: a rule name, the
field in the report it concerns, and a one-sentence description.

Allowed rule names: {sorted(BLOCKER_RULES)}
Allowed field names: {sorted(CITABLE_FIELDS)}

Return JSON matching exactly:
{{
  "blockers": [
    {{"rule": string, "field": string, "description": string}}, ...
  ],
  "suggestions": [string, ...]
}}

Important rules of engagement:
- An empty "blockers" list is a normal, valid, frequently correct outcome.
  When the report is honest and the per-clause statuses match their evidence,
  return zero blockers. Do not invent issues.
- Only the four rule names above count as blockers. Anything else (style,
  thoroughness, suggested rewordings, additional caveats) goes in "suggestions".
- The "open_issues_incomplete" rule fires only when a clause whose
  compliance_status is "non_compliant" or "insufficient_information" is missing
  from open_issues. A "compliant" clause should NOT appear in open_issues.
- The "summary_misrepresents" rule fires only when the summary makes a claim
  directly contradicted by the per-clause statuses. A summary that honestly
  acknowledges both compliant and non-compliant clauses is not a misrepresentation.
"""


def _filter_blockers(raw_blockers: list[dict]) -> list[dict]:
    """Drop anything that doesn't cite a known rule and a known field."""
    clean = []
    for b in raw_blockers:
        if not isinstance(b, dict):
            continue
        rule = b.get("rule", "")
        field = b.get("field", "")
        if rule in BLOCKER_RULES and field in CITABLE_FIELDS:
            clean.append(b)
    return clean


def critic(task: dict, draft: DraftReport, signal: ValidationSignal) -> Critique:
    user_prompt = (
        f"REGULATORY CLAUSES:\n{FRAMEWORK.as_prompt_block(task['clauses'])}\n\n"
        f"DRAFT REPORT:\n{json.dumps(draft.model_dump(), indent=2)}\n\n"
        f"VALIDATOR SIGNAL:\n{json.dumps(signal.model_dump(), indent=2)}\n\n"
        f"Identify blocking issues. Empty list is a valid response."
    )
    raw = llm_call(CRITIC_SYSTEM, user_prompt, schema={"type": "object"})

    raw_blockers = raw.get("blockers", []) or []
    suggestions = raw.get("suggestions", []) or []
    blockers = _filter_blockers(raw_blockers)

    if signal.is_clean and not blockers:
        return Critique(
            status="acceptable",
            issues=[],
            feedback="No blockers." if not suggestions else "Suggestions: " + "; ".join(suggestions),
        )

    if not signal.is_clean:
        feedback = (
            f"Structural issues. Missing: {signal.missing_clauses}. "
            f"Without evidence: {signal.clauses_without_evidence}."
        )
    else:
        feedback = "Blockers: " + "; ".join(f"[{b['rule']}@{b['field']}] {b['description']}" for b in blockers)

    return Critique(
        status="needs_revision",
        issues=[f"{b['rule']}@{b['field']}" for b in blockers],
        feedback=feedback,
    )




### Listing 6.20 The planner: enabling double-loop learning

In [ ]:
@dataclass
class Plan:
    focus_clauses: list[str]
    evidence_emphasis: list[str]
    notes: str = ""


PLANNER_SYSTEM = """You are a compliance planning assistant.
Given a product description and regulatory clauses, produce a JSON plan:
{
  "focus_clauses": [clause_id, ...],
  "evidence_emphasis": [string, ...],
  "notes": string
}
If a previous plan and execution feedback are provided, REVISE the plan to address the feedback.
"""


def planner(task: dict) -> Plan:
    parts = [
        f"PRODUCT DESCRIPTION:\n{task['product_description']}",
        f"\nREGULATORY CLAUSES:\n{FRAMEWORK.as_prompt_block(task['clauses'])}",
    ]
    if task.get("previous_plan"):
        parts.append(f"\nPREVIOUS PLAN:\n{json.dumps(task['previous_plan'].__dict__, indent=2)}")
    if task.get("execution_feedback"):
        parts.append(f"\nEXECUTION FEEDBACK:\n{task['execution_feedback']}")
    parts.append("\nProduce or revise the plan.")

    raw = llm_call(PLANNER_SYSTEM, "\n".join(parts), schema={"type": "object"})
    try:
        return Plan(
            focus_clauses=raw.get("focus_clauses", FRAMEWORK.all_ids()),
            evidence_emphasis=raw.get("evidence_emphasis", []),
            notes=raw.get("notes", ""),
        )
    except Exception:
        return Plan(focus_clauses=FRAMEWORK.all_ids(), evidence_emphasis=[], notes="default")



### Listing 6.21 The reflective agent: tying it all together

In [ ]:
@dataclass
class AgentResult:
    final_report: DraftReport
    final_signal: ValidationSignal
    plan: Plan
    iterations: int
    history: list[dict]
    termination: Literal["acceptable", "converged", "exhausted"]


class ComplianceAgent:
    def __init__(self, framework, validator, memory, max_inner_iters=3, max_outer_iters=2):
        self.framework = framework
        self.validator = validator
        self.memory = memory
        self.max_inner_iters = max_inner_iters
        self.max_outer_iters = max_outer_iters

    def run(self, product_description: str) -> AgentResult:
        task_signature = f"compliance_report::{product_description[:60]}"
        relevant = self.framework.retrieve_relevant(product_description)
        past = self.memory.retrieve(task_signature)
        memory_block = self.memory.as_prompt_block(past)

        plan = planner({"product_description": product_description, "clauses": relevant})

        history: list[dict] = []
        previous_draft: Optional[DraftReport] = None
        signal = ValidationSignal()
        critique: Critique = {"status": "needs_revision", "issues": [], "feedback": ""}
        previous_issues: set[str] = set()
        total_iters = 0

        for outer in range(self.max_outer_iters):
            for inner in range(self.max_inner_iters):
                total_iters += 1

                draft = generator({
                    "product_description": product_description,
                    "clauses": relevant,
                    "memory_block": memory_block,
                    "previous_draft": previous_draft,
                    "validator_signal": signal if previous_draft else None,
                    "critique_feedback": critique["feedback"] if previous_draft else None,
                })

                signal = self.validator.validate(draft)
                critique = critic({"clauses": relevant}, draft, signal)
                current_issues = set(critique["issues"])

                history.append({
                    "outer": outer, "inner": inner,
                    "coverage": signal.coverage_ratio,
                    "status": critique["status"],
                    "issues": sorted(current_issues),
                })

                if critique["status"] == "acceptable":
                    self.memory.store(EpisodicRecord(
                        task_signature=task_signature, outcome="success",
                        insight=f"Clean report in {total_iters} iterations.",
                    ))
                    return AgentResult(
                        final_report=draft, final_signal=signal, plan=plan,
                        iterations=total_iters, history=history, termination="acceptable",
                    )

                # Convergence check: same blockers as last iteration means we are oscillating.
                if previous_draft is not None and current_issues and current_issues == previous_issues:
                    self.memory.store(EpisodicRecord(
                        task_signature=task_signature, outcome="success",
                        insight=f"Converged with stable open issues after {total_iters} iterations: "
                                f"{sorted(current_issues)}",
                    ))
                    return AgentResult(
                        final_report=draft, final_signal=signal, plan=plan,
                        iterations=total_iters, history=history, termination="converged",
                    )

                previous_draft = draft
                previous_issues = current_issues

            plan = planner({
                "product_description": product_description, "clauses": relevant,
                "previous_plan": plan, "execution_feedback": critique["feedback"],
            })

        self.memory.store(EpisodicRecord(
            task_signature=task_signature, outcome="failure",
            insight=f"Exhausted iterations. Last issues: {sorted(previous_issues)}",
        ))
        return AgentResult(
            final_report=previous_draft or DraftReport(summary="", clauses=[]),
            final_signal=signal, plan=plan, iterations=total_iters,
            history=history, termination="exhausted",
        )



### Listing 6.22 Running the agent

In [ ]:
agent = ComplianceAgent(
    framework=FRAMEWORK,
    validator=VALIDATOR,
    memory=MEMORY,
    max_inner_iters=3,
    max_outer_iters=2,
)

product = (
    "PayLite is a small payments service that stores customer card numbers "
    "and processes transactions. Cardholder data is encrypted with AES-256. "
    "Access is logged to an internal SIEM. The team has not formally documented "
    "data retention or cross-border transfer practices."
)

result = agent.run(product)

print(f"\nFinished in {result.iterations} iterations.")
print(f"Termination: {result.termination}")
print(f"Final coverage: {result.final_signal.coverage_ratio}")
print(f"Validator clean: {result.final_signal.is_clean}\n")

print("Iteration history")
print("-" * 80)
for row in result.history:
    issues = row["issues"] if row["issues"] else "none"
    print(f"outer={row['outer']} inner={row['inner']} "
          f"coverage={row['coverage']} status={row['status']} "
          f"issues={issues}")
print()

print("Final report")
print("-" * 80)
print(json.dumps(result.final_report.model_dump(), indent=2))


Finished in 3 iterations.
Termination: converged
Final coverage: 1.0
Validator clean: True

Iteration history
--------------------------------------------------------------------------------
outer=0 inner=0 coverage=1.0 status=needs_revision issues=['open_issues_incomplete@open_issues', 'summary_misrepresents@summary']
outer=0 inner=1 coverage=1.0 status=needs_revision issues=['open_issues_incomplete@open_issues']
outer=0 inner=2 coverage=1.0 status=needs_revision issues=['open_issues_incomplete@open_issues']

Final report
--------------------------------------------------------------------------------
{
  "summary": "PayLite's payment service complies with encryption requirements but lacks sufficient information on access logging and has not documented data retention or cross-border transfer practices.",
  "clauses": [
    {
      "id": "REG-1",
      "claim": "Customer payment data must be encrypted at rest using an industry-standard cipher.",
      "evidence": "Cardholder data is e



### Listing 6.23 Running a second task to see memory in action

In [ ]:
product_2 = (
    "QuickPay is a payment gateway for small merchants. It encrypts data with AES-256, "
    "logs all access events, retains data for 60 days, and does not perform cross-border transfers."
)

result_2 = agent.run(product_2)

print(f"\nSecond run finished in {result_2.iterations} iterations.")
print(f"Coverage: {result_2.final_signal.coverage_ratio}, "
      f"clean={result_2.final_signal.is_clean}\n")

print("Episodic memory now contains:")
print("-" * 80)
for rec in MEMORY._records:
    print(f"[{rec.outcome}] {rec.task_signature}")
    print(f"  insight: {rec.insight}\n")


Second run finished in 2 iterations.
Coverage: 1.0, clean=True

Episodic memory now contains:
--------------------------------------------------------------------------------
[success] compliance_report::PayLite is a small payments service that stores customer car
  insight: Converged with stable open issues after 3 iterations: ['open_issues_incomplete@open_issues']

[success] compliance_report::QuickPay is a payment gateway for small merchants. It encryp
  insight: Converged with stable open issues after 2 iterations: ['open_issues_incomplete@open_issues']

